# App Review Scraper
Scrapes 1 and 2-star reviews from App Store and Play Store for the last 60 days.

**Author:** Hugo Malorey  
**Purpose:** AI PM Portfolio — competitive & product analysis

## 1. Imports & config

In [ ]:
import pandas as pd
from datetime import datetime, timedelta, timezone
from app_store_scraper import AppStore
from google_play_scraper import reviews, Sort

# Config
DAYS_BACK = 90
MIN_RATING = 1
MAX_RATING = 2
CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)

# Define App Identifiers
APP_STORE_ID = "1608783388"       # Bitstack iOS App ID (FR)
APP_STORE_NAME = "bitstack"       # App name slug
PLAY_STORE_ID = "com.bitstack.app" # Bitstack Android package name

# Language and market
COUNTRY = "fr"
LANGUAGE = "fr"

print(f"Scraping reviews from {CUTOFF_DATE.strftime('%Y-%m-%d')} to today")
print(f"Rating filter: {MIN_RATING} to {MAX_RATING} stars")

Scraping reviews from 2026-02-08 to today
Rating filter: 1 to 2 stars


## 2. App Store scraper (iOS)

In [11]:
print("Scraping App Store reviews...")

app = AppStore(
    country=COUNTRY,
    app_name=APP_STORE_NAME,
    app_id=APP_STORE_ID
)

app.review(how_many=500)

# Filter by rating and date
ios_reviews = [
    {
        "source": "App Store",
        "date": r["date"],
        "rating": r["rating"],
        "title": r.get("title", ""),
        "review": r["review"],
        "username": r.get("userName", "anonymous")
    }
    for r in app.reviews
    if r["rating"] <= MAX_RATING
    and r["date"].replace(tzinfo=timezone.utc) >= CUTOFF_DATE
]

print(f"Found {len(ios_reviews)} iOS reviews ({MIN_RATING}-{MAX_RATING} stars, last {DAYS_BACK} days)")

Scraping App Store reviews...


2026-04-09 13:52:15,718 [INFO] Base - Initialised: AppStore('fr', 'bitstack', 1608783388)
2026-04-09 13:52:15,719 [INFO] Base - Ready to fetch reviews from: https://apps.apple.com/fr/app/bitstack/id1608783388
2026-04-09 13:52:16,006 [ERROR] Base - Something went wrong: Expecting value: line 1 column 1 (char 0)
2026-04-09 13:52:16,007 [INFO] Base - [id:1608783388] Fetched 0 reviews (0 fetched in total)


Found 0 iOS reviews (1-2 stars, last 60 days)


In [13]:
import requests
import json

print("Scraping App Store reviews...")

ios_reviews = []
for page in range(1, 3):  # 10 pages max
    url = f"https://itunes.apple.com/fr/rss/customerreviews/page={page}/id={APP_STORE_ID}/sortby=mostrecent/json"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    
    if response.status_code != 200:
        break
    
    data = response.json()
    entries = data.get("feed", {}).get("entry", [])
    
    if not entries:
        break
    
    for entry in entries:
        rating = int(entry.get("im:rating", {}).get("label", 5))
        date_str = entry.get("updated", {}).get("label", "")
        date = datetime.fromisoformat(date_str[:10]).replace(tzinfo=timezone.utc)
        
        if date < CUTOFF_DATE:
            continue
        if rating > MAX_RATING:
            continue
            
        ios_reviews.append({
            "source": "App Store",
            "date": date,
            "rating": rating,
            "title": entry.get("title", {}).get("label", ""),
            "review": entry.get("content", {}).get("label", ""),
            "username": entry.get("author", {}).get("name", {}).get("label", "anonymous")
        })

print(f"Found {len(ios_reviews)} iOS reviews ({MIN_RATING}-{MAX_RATING} stars, last {DAYS_BACK} days)")

Scraping App Store reviews...
Found 20 iOS reviews (1-2 stars, last 60 days)


## 4. Play Store scraper (Android)

In [14]:
print("Scraping Play Store reviews...")

android_reviews = []

for star in range(MIN_RATING, MAX_RATING + 1):
    result, _ = reviews(
        PLAY_STORE_ID,
        lang=LANGUAGE,
        country=COUNTRY,
        sort=Sort.NEWEST,
        count=250,
        filter_score_with=star
    )

    filtered = [
        {
            "source": "Play Store",
            "date": r["at"],
            "rating": r["score"],
            "title": "",
            "review": r["content"],
            "username": r.get("userName", "anonymous")
        }
        for r in result
        if r["at"].replace(tzinfo=timezone.utc) >= CUTOFF_DATE
    ]

    android_reviews.extend(filtered)
    print(f"  {star}★: {len(filtered)} reviews found")

print(f"Total Android reviews: {len(android_reviews)}")

Scraping Play Store reviews...
  1★: 33 reviews found
  2★: 11 reviews found
Total Android reviews: 44


## 5. Merge & clean

In [15]:
all_reviews = ios_reviews + android_reviews

df = pd.DataFrame(all_reviews)
df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("date", ascending=False).reset_index(drop=True)

print(f"\nTotal reviews: {len(df)}")
print(f"App Store: {len(df[df['source'] == 'App Store'])}")
print(f"Play Store: {len(df[df['source'] == 'Play Store'])}")
print(f"\nRating distribution:")
print(df['rating'].value_counts().sort_index())

df.head(10)


Total reviews: 64
App Store: 20
Play Store: 44

Rating distribution:
rating
1    47
2    17
Name: count, dtype: int64


,source,date,rating,title,review,username
0,Play Store,2026-04-08 04:49:40+00:00,1,,Impossible d'ouvrir l'application depuis la de...,THT PVS
1,Play Store,2026-04-06 11:24:58+00:00,2,,la synchronisation est une catastrophe! Pas au...,Lidvine Charpentier
2,Play Store,2026-04-05 13:26:47+00:00,1,,affichage du portefeuille divergent entre plus...,Jm Cds
3,App Store,2026-04-05 00:00:00+00:00,1,Nul,J’ai même pas pu essayer l’appli que mon compt...,Ynaftis
4,Play Store,2026-04-04 15:37:55+00:00,1,,mis de l'argent sur l'application et depuis el...,Florian Pontzeele
5,Play Store,2026-04-01 16:07:41+00:00,1,,Je n'arrive pas à télécharger l'appli sur mon ...,Philippe de Veyrac
6,App Store,2026-04-01 00:00:00+00:00,2,Impossible de s’inscrire,L’idée est bonne mais impossible de s’inscrire,Caps74
7,App Store,2026-03-31 00:00:00+00:00,2,N’a plus d’intérêt,Comparais à d’autre plateforme votre btc que v...,Boudo 67
8,App Store,2026-03-31 00:00:00+00:00,1,Non respect des cours du btc,"J’ai testé pendant 1 ans environ, jamais le co...",Divinelight59
9,App Store,2026-03-28 00:00:00+00:00,1,Arnaque frais,Attention aux frais EXORBITANTS sur les petite...,Htiggt


## 6. Export to CSV

In [ ]:
output_file = f"bitstack_reviews_{datetime.now().strftime('%Y%m%d')}.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved to {output_file}")

# Preview text for Claude analysis
print("\n--- REVIEW TEXT PREVIEW (first 5) ---")
for _, row in df.head(5).iterrows():
    print(f"[{row['source']} | {row['rating']}★ | {row['date'].strftime('%Y-%m-%d')}]")
    print(f"{row['review']}")
    print()

## 7. Format for Claude analysis
Formats all reviews into a single string ready to paste into Claude.

In [17]:
lines = []
for _, row in df.iterrows():
    lines.append(
        f"[{row['source']} | {row['rating']}★ | {row['date'].strftime('%Y-%m-%d')}]\n"
        f"{row['review']}\n"
    )

claude_input = "\n".join(lines)

print(f"Total characters: {len(claude_input)}")
print("\nPaste the variable 'claude_input' into Claude for thematic analysis.")
print("\n--- SAMPLE ---")
print(claude_input)

Total characters: 15101

Paste the variable 'claude_input' into Claude for thematic analysis.

--- SAMPLE ---
[Play Store | 1★ | 2026-04-08]
Impossible d'ouvrir l'application depuis la dernière mise à jour

[Play Store | 2★ | 2026-04-06]
la synchronisation est une catastrophe! Pas automatique, les transactions apparaissent disparaissent...

[Play Store | 1★ | 2026-04-05]
affichage du portefeuille divergent entre plusieurs onglets dans l'application... question posée mais en attente de leur réponse ... étrange !!! Après contact avec le service client, ils sont dans l'impossibilité d'expliquer cette différence sur votre trésorerie Aux vues de votre réponse généraliste, vous n'en savez strictement rien d'où cela vient. Pour votre information, je mets mes applications à jour dès qu'il y a une, j'ai une excellente connexion internet et aucun VPN

[App Store | 1★ | 2026-04-05]
J’ai même pas pu essayer l’appli que mon compte est bloqué, merci beaucoup et j’ai aucun moyen de le récupéré

[Play